# 08. 자료 찾아 답하기 (RAG) — 임베딩 · 유사도 · 프롬프트

> LLM 은 모르는 것도 그럴듯하게 지어낸다(환각). 그래서 **① 질문과 가까운 자료 조각을 찾아서 ② 그 조각을 보고 답하게** 한다.
> 이것이 RAG(Retrieval-Augmented Generation, 찾아서 보강한 생성)다. 회사 규정 · 매뉴얼 Q&A 봇이 이렇게 만들어진다.

### 이 시간에 배우는 것
1. **토큰화** — 글이 숫자가 되는 과정
2. **임베딩** — 글의 뜻을 숫자 384개로 (multilingual-e5-small, INT8)
3. **코사인 유사도** — 뜻이 얼마나 가까운가
4. 글 자르기(chunking), 근거 점수로 "모른다" 고 말하게 하기, 프롬프트 만들기

### 전체 흐름
```
[준비]  자료 글 ─자르기─▶ 조각들 ─임베딩─▶ 조각 벡터들 (384개 숫자 × 조각 수)
[질문]  질문 ─임베딩─▶ 질문 벡터 ─코사인 유사도─▶ 가까운 조각 2개
        (점수가 낮으면 "자료에 없어요")  ─▶ 조각 + 질문으로 프롬프트 ─▶ LLM ─▶ 답 + 근거
```

## 1. 임베딩 모델 — 토크나이저도 OpenVINO 모델이다
`openvino_tokenizers` 를 import 하면 "글 → 토큰 번호" 계산도 OpenVINO 모델로 돌릴 수 있다. 그래서 파일이 둘이다:
`openvino_tokenizer.xml` (글 → 번호), `openvino_model.xml` (번호 → 벡터).

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import openvino_tokenizers                       # 토크나이저 연산을 등록한다 — import 만 해도 된다
tcore = ov.Core()                                # 확장은 import 뒤에 만든 Core 에 붙는다

E = MODELS / "embed" / "e5-small-int8"
tok = tcore.compile_model(E / "openvino_tokenizer.xml", "CPU")
emb = tcore.compile_model(E / "openvino_model.xml", "CPU")
print("토크나이저 출력:", [o.any_name for o in tok.outputs])
print("임베딩 모델 입력:", [i.any_name for i in emb.inputs])

## 2. 토큰화 — 글을 조각 번호로
모델은 글자를 모른다. 글을 **토큰**(자주 쓰는 글 조각)으로 자르고 번호로 바꾼다. `attention_mask` 는 진짜 토큰은 1, 길이를 맞추려 채운 빈칸은 0.

In [ ]:
t = tok(["passage: 공방은 평일 오전 9시에 연다", "query: 몇 시에 열어요?"])
print("input_ids 모양:", t["input_ids"].shape)
print("첫 문장 번호:", t["input_ids"][0])
print("첫 문장 mask:", t["attention_mask"][0])

## 3. 임베딩 — 뜻을 숫자 384개로
1. 토큰마다 384개 숫자가 나온다 → `[문장 수, 토큰 수, 384]`
2. **평균 풀링**: 진짜 토큰(mask=1)만 평균 내서 문장 하나 = 384개 숫자
3. **길이 1 로 맞추기**(정규화): 그러면 두 벡터의 내적이 곧 **코사인 유사도**

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert\,\lVert b \rVert} \quad (\text{길이가 1 이면 } a \cdot b)$$

1 에 가까울수록 뜻이 비슷하다. e5 모델은 자료 앞에 `passage: `, 질문 앞에 `query: ` 를 붙여야 제 실력이 나온다.

In [ ]:
def embed(texts, kind="passage"):
    """글 목록 → [개수, 384] (길이 1)."""
    t = tok([f"{kind}: {s}" for s in texts])
    feed = {"input_ids": t["input_ids"], "attention_mask": t["attention_mask"]}
    if "token_type_ids" in {i.any_name for i in emb.inputs}:
        feed["token_type_ids"] = np.zeros_like(t["input_ids"])
    out = emb(feed)[emb.output(0)]                    # [개수, 토큰 수, 384]
    m = t["attention_mask"][..., None].astype(np.float32)
    v = (out * m).sum(1) / m.sum(1)                    # 진짜 토큰만 평균 (mean pooling)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

S = ["고양이가 소파에서 잔다", "새끼 고양이가 낮잠을 잔다", "강아지가 공원에서 뛴다", "오늘 코스피가 올랐다"]
v = embed(S)
print("벡터 모양:", v.shape, "| 길이:", np.round(np.linalg.norm(v, axis=1), 3))
print("유사도 표:")
for i, s in enumerate(S):
    print(f"  {s:18s}", " ".join(f"{x:5.2f}" for x in v @ v.T[:, i]))

## 4. 자료 — 조각으로 나누기
긴 글을 통째로 임베딩하면 뜻이 뭉개진다. **문단 단위**로 자르고, 긴 문단은 다시 자른다.
- 너무 작게 자르면: 조각 하나에 답이 다 안 들어간다
- 너무 크게 자르면: 상관없는 말이 섞여 점수가 흐려진다

In [ ]:
DOC = """
공방 이용 안내

공방은 평일 오전 9시부터 오후 6시까지 연다. 토요일은 오후 1시까지이고 일요일과 공휴일은 쉰다.

3D 프린터는 하루 최대 3시간까지 예약할 수 있다. 예약은 공방 입구의 태블릿에서 하고, 10분 넘게 늦으면 예약이 취소된다.

레이저 커터는 안전 교육을 들은 사람만 쓸 수 있다. 안전 교육은 매주 화요일 오후 4시에 30분 동안 한다.

납땜은 환기 장치를 켠 작업대에서만 한다. 납땜 인두는 쓰고 나서 반드시 거치대에 꽂고 전원을 끈다.

재료는 기본 필라멘트(PLA)와 합판 3mm 를 무료로 쓸 수 있다. 다른 재료는 가져와서 쓴다.
"""

def split(text, size=200):
    """빈 줄(문단)로 나누고, 긴 문단은 size 글자 안쪽에서 띄어쓰기 자리로 자른다."""
    out = []
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        while len(para) > size:
            cut = para.rfind(" ", 0, size)
            cut = cut if cut > 0 else size
            out.append(para[:cut]); para = para[cut:].strip()
        out.append(para)
    return out

chunks = split(DOC)
V = embed(chunks, "passage")                         # 자료 조각 벡터 — 한 번만 만들어 둔다
for i, c in enumerate(chunks):
    print(i, c[:40], "...")
print("조각 벡터:", V.shape)

## 5. 질문과 가까운 조각 찾기
질문 벡터와 모든 조각 벡터의 내적 = 유사도. 큰 순서로 `k` 개.
"주차장" 처럼 **자료에 없는 질문**은 가장 높은 점수도 낮게 나온다 — 이것을 이용해 "모른다" 고 답하게 한다.

In [ ]:
def search(q, k=2):
    s = V @ embed([q], "query")[0]                    # 길이가 1이라 내적 = 코사인 유사도
    order = np.argsort(-s)[:k]
    return [(float(s[i]), chunks[i]) for i in order]

for q in ["토요일에도 열어요?", "레이저 커터 쓰려면 어떻게 해요?", "주차장 있어요?"]:
    print("질문:", q)
    for score, c in search(q):
        print(f"   {score:.3f}  {c[:50]}")

## 6. 찾은 조각으로 답하기 — LLM
VLM(Gemma3 4B)을 글만 넣어 쓴다. **3.5GB** 라 처음 올리는 데 시간이 걸린다 (edge-lab 이 켜져 있으면 끄고 한다).

**프롬프트**는 "자료만 보고 답하라" 를 분명히 하고, 자료 → 질문 → "답:" 순서로 둔다.
`do_sample=False` 로 같은 질문에는 같은 답이 나오게 한다.

In [ ]:
import openvino_genai as og

VLM_DIR = MODELS / "vlm" / "gemma3-4b-int4"
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
t0 = time.perf_counter()
llm = og.VLMPipeline(str(VLM_DIR), DEVICE)
print(f"LLM 올림: {time.perf_counter() - t0:.0f}초 ({DEVICE})")

cfg = og.GenerationConfig()
cfg.max_new_tokens = 160
cfg.do_sample = False
cfg.repetition_penalty = 1.15

In [ ]:
MIN_SCORE = 0.60        # 이보다 낮으면 자료에 없는 질문으로 본다 (e5-small INT8 한국어 기준으로 맞춘 값)

def make_prompt(q, hits):
    context = "\n".join(c for _, c in hits)
    return (f"아래 자료만 보고 질문에 한국어로 짧게 답하세요.\n--- 자료 ---\n{context}\n--- 끝 ---\n"
            f"질문: {q}\n답:")

def answer(q):
    hits = search(q, 2)
    if hits[0][0] < MIN_SCORE:
        return "자료에서 찾지 못했어요.", hits
    return str(llm.generate(make_prompt(q, hits), generation_config=cfg)).strip(), hits

print("— LLM 에 실제로 들어가는 글 —")
print(make_prompt("안전 교육은 언제 해요?", search("안전 교육은 언제 해요?", 2)))

In [ ]:
for q in ["3D 프린터는 몇 시간까지 쓸 수 있어요?", "안전 교육은 언제 해요?", "주차장 있어요?"]:
    t0 = time.perf_counter()
    a, hits = answer(q)
    print(f"Q: {q}\nA: {a}\n   (근거 점수 {hits[0][0]:.2f} · {time.perf_counter() - t0:.1f}초)\n")

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| `Cannot create ... SpecialTokensSplit` | `import openvino_tokenizers` 를 빠뜨렸거나, 그 전에 만든 Core 를 썼다 |
| 점수가 다 비슷하게 낮다 | `passage: ` / `query: ` 를 안 붙였다 |
| 맞는 질문인데 "못 찾았어요" | `MIN_SCORE` 가 너무 높다 → 점수를 찍어 보고 조정 |
| 답이 자료와 다르다 | 프롬프트에 "자료만 보고" 가 없거나 조각이 너무 크다 |
| 메모리 부족 | edge-lab 이 같은 모델을 올려 둔 상태 → 끄고 다시 |

## 정리
- 임베딩 = 뜻을 담은 벡터. 길이 1 로 맞추면 내적 = 코사인 유사도.
- RAG = 찾기(임베딩 · 유사도) + 답하기(LLM). **찾기 점수**로 "모른다" 를 판단할 수 있다.
- 조각 크기 · 기준 점수 · 프롬프트가 품질을 가른다.

---
# 🛠 미니 프로젝트 — 내 문서 Q&A 봇

**만들 것**: 텍스트 파일 하나를 읽어 두고, 질문을 입력하면 답과 **근거 조각**을 함께 보여 주는 봇. 빈 줄을 입력하면 끝난다.

**쓰는 것**: `split` · `embed` · `search` · `answer` (이 노트북에서 만든 함수 그대로)

**순서**
1. 문서를 읽는다 — `my_doc.txt` 가 있으면 그것, 없으면 위의 `DOC`
2. 조각 · 벡터를 만든다
3. 질문 → 답 + 근거 + 점수 를 반복

In [ ]:
doc_path = Path("my_doc.txt")                     # 이 노트북 옆에 내 문서를 두면 그것을 쓴다
text = doc_path.read_text(encoding="utf-8") if doc_path.exists() else DOC
chunks = split(text)
V = embed(chunks, "passage")
print(f"문서: {doc_path if doc_path.exists() else '예제 DOC'} · 조각 {len(chunks)}개")

while True:
    q = input("질문 (끝내려면 그냥 Enter): ").strip()
    if not q:
        break
    a, hits = answer(q)
    print(f"\n답: {a}")
    for score, c in hits:
        print(f"  근거 {score:.2f} | {c[:70]}")
    print()

**더 해 보기**
- 답과 근거를 `qa_log.txt` 에 질문마다 덧붙여 저장하기
- 근거 조각에서 질문 낱말을 `[ ]` 로 표시해 보여 주기
- 문서를 여러 개(폴더 안 `.txt` 전부) 읽고, 근거에 파일 이름도 보여 주기